In [1]:
import pandas as pd
import numpy as np

In [ ]:
INPUT_FILE = "keystroke_log.csv"  
OUTPUT_FILE = "feature.csv" 
WINDOW_SIZE = 50 # how many rows are allowed in window
SLIDE_STEP = 10 # how many steps should the sliding window slide
MAX_VALID_TIME = 2.0 # whats the max valid dwell and flight time


In [ ]:
def load_and_clean(path):
    df = pd.read_csv(path)
    df = df.sort_values("press_time").reset_index(drop=True) # sort according to press time
    df["dwell"] = df["release_time"]-df["press_time"] # dwell time is how much time the key is pressed for
    df = df[(df["dwell"]>0) & (df["dwell"]< MAX_VALID_TIME)] # dwell time should be greater than 0 and less than 2
    df = df.reset_index(drop=True) # reset indices again to drop garbage indices

    df["flight"] = df["press_time"].shift(-1) - df["release_time"]
    # shift the press time one row up that way we can minus the previous release time from the next press time
    # yileding us the flight time, or the the time between the last and next key press 
    df.loc[len(df)-1, "flight"] = np.nan
    # because we shifted the prwess timees by one up, the last index is empty now so that empty - release time
    # of previous yields garbage value so, we initialize it to nan using loc

    return df # returning the cleaned dataframe

In [ ]:
def extract_features(window): # giving each window as the input
    duration = window["release_time"].iloc[-1] - window["press_time"].iloc[0]
    # duration is the the time diff between the first key pressed and the last key released. [-1] gives last index
    n_keys = len(window)
    # gives how many keys/rows are there in the window

    valid_flights = window["flight"].dropna()
    # drops nan value at the end of the flight column, if it exists in the particular window
    valid_flights = valid_flights[(valid_flights >=0 ) & (valid_flights<MAX_VALID_TIME)]
    # 
    backspace_count = (window["key_id"] == "Key.backspace").sum()
    # find backspace count of each window. how many backspaces are pressed in each row of the window then sums it.
 
    return { # returns a dictionary object with the following keys
        "avg_dwell": window["dwell"].mean(),
        "std_dwell": window["dwell"].std(),
        "avg_flight": valid_flights.mean() if len(valid_flights) > 0 else 0.0, 
        "std_flight": valid_flights.std() if len(valid_flights) > 0 else 0.0,
        "typing_speed": n_keys / duration if duration > 0 else 0.0, 
        # typing speed of the window is no. of keys/duration. gives key per sec
        "backspace_rate": backspace_count / n_keys, 
        # no. of backspaces / no. of total keys -> give how many % of backspaces pressed in each window
    }

In [ ]:
def build_feature_table(df, window_size = WINDOW_SIZE, slide_step = SLIDE_STEP):
    rows = [] # empty list
    start = 0 # we start from zeroth index of the df

    while start+window_size<=len(df): # example start = 60 and window_size = 50, len(df) = 100. loop breaks
        window = df.iloc[start :  start+window_size]
        # window will have rows of the df from start to start+window_size, initiall from 0 to 0+50 -> 0 to 50
        # it will only store til l to 49 tho, as in python slicing, last index is not included
        rows.append(extract_features(window))
        # append output of each window on the given function in the empty list called row
        start += slide_step  # increase start by slide_step amount ( here 10)

    return pd.DataFrame(rows) # return the list of dictionaries and convert them into a dataframe directly for future use